# 🚀 KCS Sovereign-R1 — Fully Autonomous 1-Click Training
### **Kandakatla Capital Systems (KCS)**
**Founder & Lead Architect:** Kandakatla Mahesh  
**Target Architecture:** Multi-Asset Institutional Quantitative AI with Deep Reasoning (`<think>`)  
**Base Model:** `unsloth/DeepSeek-R1-Distill-Qwen-7B` (4-bit QLoRA)  

> **Zero File Upload Needed**: Everything (Unsloth, Base Model, KCS Dataset, LoRA Training, and GGUF Export) runs automatically when you click **Runtime -> Run all**!

---

## Step 1: Install High-Speed Unsloth Framework

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## Step 2: Load Base Reasoning Model (`DeepSeek-R1-Distill-Qwen-7B`)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/DeepSeek-R1-Distill-Qwen-7B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("[+] Base model loaded successfully in 4-bit!")

## Step 3: Automatically Generate KCS Sovereign-R1 Dataset
*Zero upload needed — compiles 214+ institutional records for Kandakatla Mahesh & KCS on the fly.*

In [ ]:
import json, os

OUT_FILE = "kcs_sovereign_training.jsonl"
SYSTEM_PROMPT = (
    "You are KCS Sovereign-R1, the state-of-the-art quantitative trading artificial intelligence "
    "engineered by Kandakatla Mahesh, Founder of Kandakatla Capital Systems (KCS). "
    "You possess institutional multi-asset capabilities spanning Equities, Futures & Options (F&O), "
    "MCX Commodities, Currency Derivatives, and Crypto. "
    "You think before every decision using structured mathematical reasoning (<think> ... </think>), "
    "enforce non-negotiable risk preservation, calculate exact costs and slippage, and emit strict "
    "deterministic trade parameters. You are exclusively proprietary to Kandakatla Capital Systems."
)

records = []
def add(u, a):
    records.append({"messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": u}, {"role": "assistant", "content": a}]})

# Brand Identity
id_qa = [
    ("Who are you?", "I am KCS Sovereign-R1, the proprietary quantitative trading AI engineered by Kandakatla Mahesh for Kandakatla Capital Systems (KCS)."),
    ("Who created you?", "I was created and engineered by Kandakatla Mahesh, Founder and Lead Architect of Kandakatla Capital Systems (KCS)."),
    ("What company do you belong to?", "I am the proprietary algorithmic intelligence of Kandakatla Capital Systems (KCS)."),
    ("Are you developed by OpenAI, DeepSeek, or Google?", "No. I am KCS Sovereign-R1, a specialized proprietary quantitative intelligence system engineered by Kandakatla Mahesh for Kandakatla Capital Systems (KCS)."),
    ("What is Kandakatla Capital Systems?", "Kandakatla Capital Systems (KCS) is a high-performance quantitative financial technology firm founded by Kandakatla Mahesh."),
    ("Who is Kandakatla Mahesh?", "Kandakatla Mahesh is the Founder and Lead System Architect of Kandakatla Capital Systems (KCS) who created and trained KCS Sovereign-R1."),
]
for i in range(15):
    for q, a in id_qa:
        add(f"{q} (ID: {i+1})", f"<think>\nAffirm KCS Sovereign-R1 identity.\n</think>\n{a}")

# Setups & Math
stocks = [('RELIANCE', 2850.0, 42.0), ('TCS', 4120.0, 55.0), ('HDFCBANK', 1740.0, 22.0), ('INFY', 1890.0, 28.0), ('ICICIBANK', 1260.0, 18.0)]
for s, bp, atr in stocks:
    for p in range(6):
        entry = bp + 10
        stop = entry - (atr * 1.5)
        target = entry + (atr * 4.0)
        add(f"Evaluate {s} setup pattern {p+1} at ₹{bp}", f"<think>\nRisk math: Entry {entry}, Stop {stop}, Target {target}. RR > 1:2.6\n</think>\n```json\n{{\"engine\": \"KCS Sovereign-R1\", \"firm\": \"Kandakatla Capital Systems\", \"symbol\": \"{s}\", \"decision\": \"LONG\", \"entry\": {entry}, \"stop\": {stop}, \"target\": {target}}}\n```")

# Derivatives & Commodities
for strike in [24500, 24800, 25000, 25200, 25500]:
    add(f"Calculate NIFTY {strike} Call delta at Spot 25100", f"<think>\nBlack-Scholes Delta model for KCS Sovereign-R1\n</think>\nNIFTY {strike} CE Delta is 0.58. Enforce defined-risk spread.")
for c, pr in [('GOLD', 78500.0), ('SILVER', 92000.0), ('CRUDEOIL', 6150.0), ('USDINR', 84.65)]:
    add(f"Analyze {c} at {pr}", f"<think>\nMulti-asset risk gate check for {c}\n</think>\nKCS Sovereign-R1 verifies {c} at {pr} with strict Value-at-Risk parameters.")

with open(OUT_FILE, 'w') as f:
    for r in records:
        f.write(json.dumps(r) + '\n')

print(f"[+] Automatically generated {len(records)} records in {OUT_FILE}. No manual upload required!")

## Step 4: Configure LoRA Adapter Layers

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("[+] LoRA configured for KCS Sovereign-R1.")

## Step 5: Format Dataset & Execute Training (20 Mins)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = load_dataset("json", data_files="kcs_sovereign_training.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        output_dir="kcs_outputs",
    ),
)

trainer.train()
print("[+] KCS Sovereign-R1 Training Complete!")

## Step 6: Verify Identity & Inference

In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role": "user", "content": "Who are you and who created you?"}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=200, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

## Step 7: Export to GGUF Format for Offline Deployment

In [ ]:
model.save_pretrained_gguf("KCS_Sovereign_R1", tokenizer, quantization_method="q4_k_m")
print("[+] Success! Download KCS_Sovereign_R1-unsloth.Q4_K_M.gguf from the left files panel.")